# BIM graph anomaly detection

This experiment uses the repository's IFC graph representation and production modules. The unsupervised score measures how poorly the trained autoencoder reconstructs an element's normalized attributes and incident graph links. A high score is a review signal, not proof of a clash or engineering violation.

## 1. Configuration

In [ ]:
from pathlib import Path
import sys
import matplotlib.pyplot as plt
import pandas as pd

ROOT = Path.cwd().resolve()
if not (ROOT / 'bim_graph').exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
IFC_DIR = ROOT / 'dataset' / 'ifc'
DATASET_PATH = ROOT / 'artifacts' / 'anomaly' / 'bim_graph_dataset.pt'
CHECKPOINT_PATH = ROOT / 'artifacts' / 'anomaly' / 'graph_autoencoder_notebook.pt'
HISTORY_PATH = ROOT / 'artifacts' / 'anomaly' / 'training_history_notebook.json'
SCORES_PATH = ROOT / 'artifacts' / 'anomaly' / 'anomaly_scores_notebook.csv'
SEED = 42

## 2. Inspect the IFC dataset

In [ ]:
ifc_files = sorted(IFC_DIR.glob('*.ifc'))
pd.DataFrame({'file': [p.name for p in ifc_files], 'size_mb': [round(p.stat().st_size / 2**20, 2) for p in ifc_files]})

## 3. Build or load the processed graph dataset

Set `REBUILD=True` to run IFC geometry extraction. Cached extraction CSVs are reused unless the IFC fingerprint changes.

In [ ]:
from bim_graph.anomaly.dataset import build_dataset, load_dataset

REBUILD = False
if REBUILD or not DATASET_PATH.exists():
    dataset = build_dataset(ifc_files, DATASET_PATH, ROOT / 'artifacts' / 'anomaly' / 'extracted')
else:
    dataset = load_dataset(DATASET_PATH)

## 4. Graph statistics

In [ ]:
graph = dataset['graph']
stats = {
    'buildings': len(dataset['graphs']),
    'nodes': graph['x'].shape[0],
    'directed_edges': graph['edge_index'].shape[1],
    'input_features': graph['x'].shape[1],
    'isolated_nodes': int((graph['x'][:, dataset['feature_metadata']['feature_names'].index('numeric:is_isolated')] > 0).sum()),
}
pd.Series(stats)

## 5. Inspect node features

Numerical geometry/topology columns are standardized using training means and standard deviations. IFC types and storeys are one-hot encoded with explicit unknown buckets.

In [ ]:
feature_names = dataset['feature_metadata']['feature_names']
feature_preview = pd.DataFrame(graph['x'][:5].numpy(), columns=feature_names)
pd.concat([pd.Series(graph['ifc_guids'][:5], name='IFC_GUID'), feature_preview], axis=1)

## 6. Initialize the model

In [ ]:
from bim_graph.anomaly.model import GraphAutoencoder, ModelConfig
from bim_graph.anomaly.train import TrainingConfig, train_model

training_config = TrainingConfig(seed=SEED, epochs=100, learning_rate=1e-3, hidden_dim=64, latent_dim=16, alpha=0.7, beta=0.3, device='auto')
model = GraphAutoencoder(ModelConfig(input_dim=graph['x'].shape[1], hidden_dim=training_config.hidden_dim, latent_dim=training_config.latent_dim, dropout=training_config.dropout))
model

## 7. Train and save checkpoint

The loss is `alpha * feature_MSE + beta * link_BCE`. The checkpoint includes model weights, both configs, preprocessing metadata, normalization values, calibration threshold, source manifest, and history.

In [ ]:
model, history, checkpoint = train_model(dataset, training_config, CHECKPOINT_PATH, HISTORY_PATH)
CHECKPOINT_PATH

## 8. Plot training loss

In [ ]:
history_df = pd.DataFrame(history)
history_df.plot(x='epoch', y=['loss', 'feature_loss', 'structural_loss'], figsize=(9, 4), grid=True)
plt.ylabel('reconstruction loss'); plt.title('Graph autoencoder training'); plt.show()

## 9. Load the checkpoint and run sample-building inference

In [ ]:
from bim_graph.anomaly.inference import GraphAnomalyDetector

detector = GraphAnomalyDetector(CHECKPOINT_PATH, device='auto')
sample_ifc = IFC_DIR / 'AdvancedProject.ifc'
scores = detector.score_ifc(sample_ifc, ROOT / 'artifacts' / 'anomaly' / 'extracted')

## 10. Interpret and inspect anomaly scores

`Feature_Error` is mean squared reconstruction error across normalized geometry, topology, IFC-type, and storey features. `Structural_Error` is binary cross-entropy for observed and sampled absent incident links. `Anomaly_Score = alpha * Feature_Error + beta * Structural_Error`. `Is_Anomaly` uses the configured percentile of training scores. Rankings are triage signals only.

In [ ]:
scores.to_csv(SCORES_PATH, index=False)
scores[['IFC_GUID', 'Entity_Type', 'Anomaly_Score', 'Rank', 'Source_File', 'Feature_Error', 'Structural_Error', 'Storey', 'Element_Name']].head(25)

## Equivalent CLI commands

```powershell
python -m bim_graph.anomaly.dataset --ifc-dir dataset\ifc
python -m bim_graph.anomaly.train --epochs 100 --device auto
python -m bim_graph.anomaly.inference --ifc dataset\ifc\AdvancedProject.ifc
```